<div style="background:linear-gradient(90deg,#C8102E 0%,#7A0019 100%); color:white; padding:22px 28px; border-radius:8px">
  <div style="font-size:0.9em; letter-spacing:2px; opacity:0.85">NOTEBOOK 01 · DATA FOUNDATION · COMPLETED</div>
  <div style="font-size:2.0em; font-weight:700; margin-top:4px">🧱 The OMOP data foundation</div>
  <div style="font-size:1.1em; margin-top:8px; max-width:880px; opacity:0.95">
    Confirm and profile the 6 OMOP tables the shared foundation already stood up. 300
    breast-cancer patients with planted trial cohorts and a deliberate "biomarkers hide in
    the notes" gap. You read these tables; you do not create them.
  </div>
</div>

## 🗂️ What is OMOP, and what are we building?

The **OMOP Common Data Model (CDM)** is a standard way to organize health records so the same
query works across hospitals and research networks. The shared foundation already stood up
**6 of those tables** with 300 synthetic breast-cancer patients, following the OMOP CDM (a
public open standard). This notebook reads them read-only from your `source_schema`
(synthetic default `clinops_foundation`), so every query you write here runs unchanged against
any OMOP-conformant source later: just point `source_catalog` / `source_schema` at your own
OMOP tables in `databricks.yml`.

| Table | Role in this demo |
|---|---|
| `person` | One row per patient: `person_id`, `year_of_birth`, `gender_source_value`. 300 rows. |
| `condition_occurrence` | Breast-cancer diagnoses. `condition_source_value = 'Malignant neoplasm of breast'`. |
| `measurement` | **Structured** biomarker results (HER2 / ER / PR), `value_source_value` = `'Positive'`/`'Negative'`. Absent for the notes-only patients. |
| `observation` | Menopausal status (Trial B) and AJCC tumor stage. |
| `drug_exposure` | Prior therapy. Trastuzumab / Pertuzumab flag prior anti-HER2 treatment, a Trial A disqualifier. |
| `note` | **Free-text** pathology reports in `note_text`, where the hidden biomarkers live. 300 rows. |

## 🎯 Three biomarker-source groups, the whole point of the demo

We plant every patient into one of three groups based on **where their biomarkers live**. This
is what lets notebooks 04 to 07 prove the NLP value story.

<div style="display:flex; gap:14px; flex-wrap:wrap; margin-top:8px">
  <div style="flex:1; min-width:230px; background:#E8F5E9; border-radius:6px; padding:14px">
    <div style="font-size:1.6em; font-weight:700; color:#2E7D32">180</div>
    <b>both-agree</b> (person 1 to 180)<br>biomarkers in <i>both</i> <code>measurement</code> and <code>note_text</code>.
    <br>→ the NLP <b>ground truth</b>.
  </div>
  <div style="flex:1; min-width:230px; background:#FFEBEE; border-radius:6px; padding:14px">
    <div style="font-size:1.6em; font-weight:700; color:#C62828">60</div>
    <b>notes-only</b> (person 181 to 240)<br>biomarkers <i>only</i> in <code>note_text</code>.
    <br>→ <b>invisible to SQL</b>; recovered only by NLP.
  </div>
  <div style="flex:1; min-width:230px; background:#E3F2FD; border-radius:6px; padding:14px">
    <div style="font-size:1.6em; font-weight:700; color:#1565C0">60</div>
    <b>structured-only</b> (person 241 to 300)<br>biomarkers only in <code>measurement</code>.
  </div>
</div>

In [0]:
%run ./_config

# ⚙️ Configuration: Clinical Trial Pre-Screening (PRE-BUILT)

<div style="background:#f4f6f9; border-left:6px solid #C8102E; padding:14px 18px; border-radius:4px; font-size:0.95em">
This is the <b>companion config notebook</b>. It is <b>pre-built; you do not edit it</b>.
Every other notebook starts with <code>%run ./_config</code> so they all share one
catalog / schema / warehouse and the same read-only OMOP source.<br>
Just set the widgets at the top of <code>00_START_HERE</code> (matching your
bundle's <code>client_catalog</code> / <code>client_schema</code> / <code>warehouse_id</code>
/ <code>source_schema</code>) and re-run.
</div>

Everything here is Unity-Catalog-scoped (no hive_metastore) and reads from widgets, no
hardcoded secrets.

ℹ️ Not creating catalog clinops_catalog (likely pre-provisioned / no CREATE CATALOG): (com.databricks.sql.managedcatalog.acl.UnauthorizedAccessException) PERMISSION_D
✅ Writing to clinops_catalog.clinops_ml
   Reading read-only OMOP source from clinops_catalog.clinops_foundation
   SQL Warehouse: 0123456789abcdef


Helpers ready: fqn(), src(), show_md(), LLM_FAST, LLM_STRONG


## 🛠️ Where the data comes from (the shared foundation, not this kit)

The 6 OMOP tables are stood up **once** by the shared `foundation/` bundle, which lands them in
the `source_schema` (synthetic default `clinops_foundation`). Every team, including this ML
group, reads those same tables read-only and builds its own work on top. This kit does not
generate data; it **reads** the foundation's tables.

<div style="background:#E8F5E9; border-left:6px solid #2E7D32; padding:12px 16px; border-radius:4px">
To read your <b>own</b> OMOP data instead of the synthetic foundation, set
<code>source_catalog</code> / <code>source_schema</code> to your OMOP catalog and schema in
<code>databricks.yml</code>. The 6 table names are identical, so nothing downstream changes.
</div>

## 🔬 Confirm and profile the 6 OMOP tables (read from `source_schema`)

These are read-only checks. `_config` pointed Spark at your writable schema; here we switch the
session to the read-only `source_schema` so the bare table names below resolve to the
foundation's OMOP tables. Notebooks 02+ read the same source tables in their pipelines.

In [0]:
# Read-only exploration only. We do NOT create anything in the source schema.
# Use the SOURCE catalog+schema so bare table names resolve to your own OMOP too. Your
# OMOP source is typically a different catalog, so switching only the schema would miss it.
spark.sql(f"USE CATALOG {SOURCE_CATALOG}")
spark.sql(f"USE SCHEMA {SOURCE_SCHEMA}")
print(f"Reading the 6 OMOP tables from {SOURCE_CATALOG}.{SOURCE_SCHEMA}")

Reading the 6 OMOP tables from clinops_catalog.clinops_foundation


In [0]:
%sql
SELECT 'person' AS tbl, COUNT(*) AS n FROM person
UNION ALL SELECT 'condition_occurrence', COUNT(*) FROM condition_occurrence
UNION ALL SELECT 'measurement',          COUNT(*) FROM measurement
UNION ALL SELECT 'observation',          COUNT(*) FROM observation
UNION ALL SELECT 'drug_exposure',        COUNT(*) FROM drug_exposure
UNION ALL SELECT 'note',                 COUNT(*) FROM note
ORDER BY tbl;

tbl,n
condition_occurrence,300
drug_exposure,383
measurement,720
note,265
observation,720
person,300


Expected: **`person` = 300**, **`note` = 300**, `condition_occurrence` ≈ 300, and `measurement`,
`observation`, `drug_exposure` all > 0. If a table is missing or empty, confirm the shared
foundation has run and that your `source_schema` points at it (synthetic default
`clinops_foundation`, or your own OMOP catalog and schema for the real data).

In [0]:
%sql
WITH has_struct AS (
  SELECT DISTINCT person_id FROM measurement
  WHERE measurement_source_value IN ('HER2/neu','Estrogen receptor','Progesterone receptor')
),
has_note AS (
  SELECT DISTINCT person_id FROM note WHERE note_source_value = 'PATHOLOGY_REPORT'
)
SELECT
  CASE
    WHEN s.person_id IS NOT NULL AND n.person_id IS NOT NULL THEN 'both-agree (structured + note)'
    WHEN s.person_id IS NULL     AND n.person_id IS NOT NULL THEN 'notes-only  ⟵ invisible to SQL'
    WHEN s.person_id IS NOT NULL AND n.person_id IS NULL     THEN 'structured-only'
    ELSE 'neither'
  END AS biomarker_source_group,
  COUNT(*) AS patients
FROM person p
LEFT JOIN has_struct s ON p.person_id = s.person_id
LEFT JOIN has_note   n ON p.person_id = n.person_id
GROUP BY ALL
ORDER BY patients DESC;

biomarker_source_group,patients
both-agree (structured + note),180
structured-only,60
notes-only ⟵ invisible to SQL,60


Expected: **both-agree ≈ 180**, **notes-only ≈ 60**, **structured-only ≈ 60**. The 60 notes-only
patients (person 181 to 240) are the ones a SQL-only pipeline silently misses, the gap that
notebooks 04 to 05 close with NLP.

## ✅ Planted-cohort validation SQL (COMPLETED)

Before building on the data, confirm the two planted trial cohorts are present. This is a gentle
warm-up for the eligibility logic written properly in notebooks 02 and 06. **Trial A** is
seeded on person 1 to 20; **Trial B** on person 31 to 50.

In [0]:
%sql
-- Structured-only baseline for Trial A. It counts patients who are HER2 Positive by a lab
-- measurement AND have a breast-cancer diagnosis AND have NOT had anti-HER2 therapy.
-- This number MISSES the notes-only patients, which is exactly the gap nb 04 closes.
SELECT COUNT(DISTINCT m.person_id) AS trial_a_eligible
FROM measurement m
JOIN condition_occurrence co
  ON m.person_id = co.person_id
 AND co.condition_source_value = 'Malignant neoplasm of breast'
WHERE m.measurement_source_value = 'HER2/neu'
  AND m.value_source_value       = 'Positive'
  AND m.person_id NOT IN (
    SELECT person_id FROM drug_exposure
    WHERE drug_source_value IN ('Trastuzumab','Pertuzumab')
  );

trial_a_eligible
109


Expected: **≥ 20** (person 1 to 20 are guaranteed; a few incidental matches are fine).
The `NOT IN (anti-HER2)` exclusion is what keeps the ineligible controls (person 21 to 30) out.

In [0]:
%sql
-- Self-join measurement to itself (one alias for ER, one for HER2), then join observation for
-- menopausal status and condition_occurrence for the breast-cancer diagnosis.
SELECT COUNT(DISTINCT er.person_id) AS trial_b_eligible
FROM measurement er
JOIN measurement her2
  ON er.person_id = her2.person_id
JOIN observation o
  ON er.person_id = o.person_id
JOIN condition_occurrence co
  ON er.person_id = co.person_id
 AND co.condition_source_value = 'Malignant neoplasm of breast'
WHERE er.measurement_source_value  = 'Estrogen receptor' AND er.value_source_value  = 'Positive'
  AND her2.measurement_source_value = 'HER2/neu'          AND her2.value_source_value = 'Negative'
  AND o.observation_source_value   = 'Menopausal status'  AND o.value_source_value   = 'Postmenopausal';

trial_b_eligible
56


Expected: **≥ 20** (person 31 to 50 are guaranteed).

In [0]:
%sql
SELECT person_id, note_title, note_text
FROM note
WHERE note_source_value = 'PATHOLOGY_REPORT'
LIMIT 1;

person_id,note_title,note_text
1,Core Needle Biopsy — Pathology Report,"PATHOLOGY REPORT — CORE NEEDLE BIOPSY PATH-524019 | 2023-01-06 SPECIMEN: Left breast lumpectomy specimen INDICATION: Palpable Left breast mass. Rule out malignancy. HISTOLOGIC FINDINGS: Diagnosis: Invasive ductal carcinoma, grade 2 Nuclear Grade: 2/3 Estimated Tumor Size: 1.6 cm Lymphovascular Invasion: Not identified RECEPTOR / BIOMARKER STATUS: Estrogen receptor: reactive, 95% of nuclei, Allred score 8 Progesterone receptor (PR): Negative (< 1%) HER2: Positive (3+ by IHC) Ki-67: 61% MOLECULAR SUBTYPE: HR+/HER2+ (luminal B-like, HER2-positive) COMMENT: Clinical correlation recommended. These findings represent a biopsy diagnosis; final staging will require complete excision and axillary lymph node evaluation. Pathologist: Jennifer Hopkins, M.D. | Meridian Cancer Center"


<div style="background:#E8F5E9; border-left:6px solid #2E7D32; padding:12px 16px; border-radius:4px">
<b>Takeaway:</b> 6 OMOP tables, 300 patients, two planted trial cohorts, and 60 patients whose
biomarkers hide in free text. Everything downstream queries these exact table and column names.
Next, <b>notebook 02</b> reshapes raw OMOP into clean silver feature views, and that's where
you write the biomarker pivot.
</div>

## ▶️ Next step
### → Open **[02_silver_feature_pipeline]($./02_silver_feature_pipeline)** to build the silver feature views.